# Food Microbiome Exposure (FME) Analysis
## Research Question
**How does exposure to food-associated microbes influence the composition and stability of the human gut microbiome?**

### What this notebook does
1. **Data Transfer** — Load all preprocessed datasets from both sources
2. **Data Mapping** — Map dietary food items to cFMD food categories
3. **FME Quantification** — Calculate a daily Food Microbiome Exposure score per fecal sample
4. **Statistical Dataset** — Assemble one clean table ready for statistical modelling

### Key structural note
> The Johnson et al. 2019 study has **34 participants**, each contributing multiple fecal samples across up to 17 study days.
> All FME scores are computed at the **fecal sample level** (one score per sample = one dietary day per person).
> Participant-level summaries are derived by aggregating across samples — **not** by treating samples as independent observations.
> The unit of analysis for statistical models is the **fecal sample**, with participant included as a random effect.


---
## Section 1 – Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 100
print("Libraries loaded.")


In [ ]:
from pathlib import Path
import sys
# Locate repo root (directory containing config.py) regardless of launch directory
_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
DATA_DIR = _root / "data"  # robust: does not depend on __file__ inside Jupyter
sys.path.insert(0, str(_root))


---
## Section 2 – Data Transfer: Load All Datasets

### Source 1 — Johnson et al. 2019 (gut microbiome + diet)
34 participants · up to 17 fecal samples each · 566 samples total

### Source 2 — Carlino et al. 2024 — cFMD (food microbiome)
3,393 food samples · 5,057 microbial taxa


In [ ]:
# ── Source 1: Gut microbiome matrices (taxa × fecal samples) ───────────────────
gut_microbiome_abundance  = pd.read_csv(DATA_DIR / "microbiome_filtered.csv", index_col=0)

# ── Source 1: Fecal sample metadata ────────────────────────────────────────────
# index = fecal sample ID (MCT.f.XXXX)
# columns: UserName (= participant), StudyDayNo, Gender, Age, BMI, ...
fecal_sample_metadata     = pd.read_csv(DATA_DIR / "sample_meta.csv",         index_col=0)

# ── Source 1: Dietary consumption matrix (food items × fecal samples) ──────────
# rows = food items, columns = fecal sample IDs (MCT.f.XXXX)
# values = consumption amount of that food on that day
dietary_consumption_matrix= pd.read_csv(DATA_DIR / "diet_raw.csv")

# ── Source 1: Daily nutrition totals ───────────────────────────────────────────
# index = fecal sample ID
daily_nutrition_totals    = pd.read_csv(DATA_DIR / "nutrition_totals.csv",    index_col=0)

# ── Source 1: Additional dietary features ──────────────────────────────────────
daily_fiber_scores        = pd.read_csv(DATA_DIR / "diet_fiber.csv")
daily_decay_scores        = pd.read_csv(DATA_DIR / "diet_decay.csv")
participant_fiber_avg     = pd.read_csv(DATA_DIR / "fiber_avg.csv")

# ── Source 2: cFMD food microbiome abundance (food taxa × food samples) ─────────
# columns: 'StudyName__ShortSampleID' format
cfmd_food_abundance       = pd.read_csv(DATA_DIR / "combined_datasets.csv",   index_col=0, low_memory=False)
cfmd_food_abundance       = cfmd_food_abundance.apply(pd.to_numeric, errors="coerce").fillna(0.0)

# ── Source 2: cFMD sample metadata ─────────────────────────────────────────────
# columns: sample_id, category, fermented/non-fermented, country, ...
cfmd_food_metadata        = pd.read_csv(DATA_DIR / "cfmd_metadata.csv")

# ── Verify participant count ────────────────────────────────────────────────────
n_participants = fecal_sample_metadata["UserName"].nunique()
n_fecal_samples= len(fecal_sample_metadata)

print(f"Participants          : {n_participants}  (expected: 34)")
print(f"Fecal samples total   : {n_fecal_samples}")
print(f"Avg samples/person    : {n_fecal_samples / n_participants:.1f}")
print()
print("Dataset sizes:")
for name, df in [
    ("gut_microbiome_abundance",   gut_microbiome_abundance),
    ("fecal_sample_metadata",      fecal_sample_metadata),
    ("dietary_consumption_matrix", dietary_consumption_matrix),
    ("daily_nutrition_totals",     daily_nutrition_totals),
    ("cfmd_food_abundance",        cfmd_food_abundance),
    ("cfmd_food_metadata",         cfmd_food_metadata),
]:
    print(f"  {name:<32} {str(df.shape)}")


---
## Section 3 – Define the Analytical Sample Set

Only fecal samples that have **both** a gut microbiome measurement **and** a dietary record
form the analytical core. We identify these 475 samples here.


In [ ]:
# ── Identify fecal sample IDs present in both gut microbiome and diet data ──────
FECAL_SAMPLE_PREFIX = "MCT.f."

gut_microbiome_sample_ids = set(gut_microbiome_abundance.columns)
diet_record_sample_ids    = set(
    c for c in dietary_consumption_matrix.columns
    if c.startswith(FECAL_SAMPLE_PREFIX)
)

# Core analytical set: samples with BOTH gut microbiome AND diet record
analytical_sample_ids = sorted(gut_microbiome_sample_ids & diet_record_sample_ids)

print(f"Gut microbiome samples         : {len(gut_microbiome_sample_ids)}")
print(f"Diet record samples            : {len(diet_record_sample_ids)}")
print(f"Analytical samples (gut∩diet)  : {len(analytical_sample_ids)}")
print()

# Participants represented in the analytical set
analytical_participants = (
    fecal_sample_metadata
    .loc[fecal_sample_metadata.index.isin(analytical_sample_ids), "UserName"]
    .nunique()
)
print(f"Participants represented       : {analytical_participants} / {n_participants}")

# Samples per participant in the analytical set
samples_per_participant = (
    fecal_sample_metadata
    .loc[fecal_sample_metadata.index.isin(analytical_sample_ids)]
    .groupby("UserName")
    .size()
    .rename("n_analytical_samples")
)
print(f"Avg analytical samples/person  : {samples_per_participant.mean():.1f}")
print(f"Min / Max                      : {samples_per_participant.min()} / {samples_per_participant.max()}")


---
## Section 4 – FME Weights from cFMD

**Goal:** Derive a microbial richness weight for each cFMD food category.

**Why richness, not total abundance?**
The `combined_datasets` matrix stores relative abundance — each sample sums to 100 regardless of food type.
Total abundance is therefore uninformative as a weight.
**Microbial richness** (= number of distinct taxa detected per food sample) reflects how microbiome-diverse each food category truly is.

**Weight formula:**
$$w_c = \frac{\text{mean\_richness}_c - \min}{\max - \min} \quad \in [0, 1]$$

Weights are derived directly from the **cFMD food abundance matrix** — not assumed or hand-coded.


In [ ]:
# ── Step 1: Map cFMD column names → food category + fermentation status ─────────
# combined_datasets columns: 'StudyName__ShortSampleID'
# cfmd_food_metadata sample_id: 'ShortSampleID'

def extract_cfmd_short_id(column_name):
    """Strip the study name prefix from a combined_datasets column name."""
    return column_name.split("__")[-1] if "__" in column_name else column_name

# Remove the one duplicate sample_id (VB15-29) before indexing
cfmd_food_metadata_dedup = cfmd_food_metadata.drop_duplicates(
    subset="sample_id", keep="first"
)
print(f"cFMD metadata after dedup: {len(cfmd_food_metadata_dedup)} rows "
      f"(removed {len(cfmd_food_metadata) - len(cfmd_food_metadata_dedup)} duplicate)")

cfmd_id_to_meta = (
    cfmd_food_metadata_dedup
    .set_index("sample_id")[["category", "fermented/non-fermented"]]
    .to_dict(orient="index")
)

# Map each column in cfmd_food_abundance to its category and fermentation status
cfmd_column_to_category   = {}
cfmd_column_to_fermented  = {}
for col in cfmd_food_abundance.columns:
    short_id = extract_cfmd_short_id(col)
    if short_id in cfmd_id_to_meta:
        cfmd_column_to_category[col]  = cfmd_id_to_meta[short_id]["category"]
        cfmd_column_to_fermented[col] = cfmd_id_to_meta[short_id]["fermented/non-fermented"]

print(f"cFMD columns mapped: {len(cfmd_column_to_category)} / {cfmd_food_abundance.shape[1]}")


In [ ]:
# ── Step 2: Compute microbial richness per food sample ──────────────────────────
# Richness = number of non-zero taxa per food sample
cfmd_taxa_richness_per_sample = (cfmd_food_abundance > 0).sum(axis=0)

# ── Step 3: Mean richness per cFMD food category ────────────────────────────────
category_richness_values = {}
for col, category in cfmd_column_to_category.items():
    if col in cfmd_taxa_richness_per_sample.index:
        category_richness_values.setdefault(category, []).append(
            cfmd_taxa_richness_per_sample[col]
        )

cfmd_category_mean_richness = pd.Series(
    {cat: np.mean(vals) for cat, vals in category_richness_values.items()},
    name="mean_taxa_richness"
).sort_values(ascending=False)

# ── Step 4: Normalize weights to [0, 1] ─────────────────────────────────────────
richness_min = cfmd_category_mean_richness.min()
richness_max = cfmd_category_mean_richness.max()
cfmd_category_weight_normalized = (
    (cfmd_category_mean_richness - richness_min) /
    (richness_max - richness_min)
).rename("fme_weight")

cfmd_weights_table = pd.concat(
    [cfmd_category_mean_richness, cfmd_category_weight_normalized], axis=1
)

print("FME weights per cFMD food category (from cFMD microbial richness):")
display(cfmd_weights_table.round(3))


In [ ]:
# ── Visualize FME weights ──────────────────────────────────────────────────────
plt.figure(figsize=(10, 5))
cfmd_category_weight_normalized.sort_values().plot(
    kind="barh", color="steelblue", edgecolor="black", linewidth=0.5
)
plt.title("FME Weight per cFMD Food Category\n(normalized microbial taxa richness from cFMD)")
plt.xlabel("Normalized Weight  [0 = least microbe-rich  →  1 = most microbe-rich]")
plt.ylabel("cFMD Food Category")
plt.tight_layout()
plt.show()


---
## Section 5 – Data Mapping: Diet Items → cFMD Food Categories

**Goal:** Map each food item in the Johnson et al. dietary records to a cFMD food category,
creating a unified representation of food-derived microbial exposure.

**Method:** Extract the L1 (top-level) taxonomy label from the `taxonomy` column of `diet_raw`
(e.g. `L1_Milk_and_Milk_Products` → `dairy`) and map it to the matching cFMD category.


In [ ]:
# ── Extract L1 food category from taxonomy string ───────────────────────────────
def extract_l1_food_category(taxonomy_string):
    """
    Extract the top-level food category (L1) from the hierarchical taxonomy string.
    Example: 'L1_Milk_and_Milk_Products;L2_...' → 'milk and milk products'
    """
    if pd.isna(taxonomy_string):
        return "unknown"
    l1_segment = str(taxonomy_string).split(";")[0]
    return l1_segment.replace("L1_", "").replace("_", " ").strip().lower()

dietary_consumption_matrix["diet_l1_category"] = (
    dietary_consumption_matrix["taxonomy"]
    .apply(extract_l1_food_category)
)

print("Unique L1 diet categories found:")
for cat, count in dietary_consumption_matrix["diet_l1_category"].value_counts().items():
    print(f"  {cat:<45} {count:>5} food items")


In [ ]:
# ── Mapping table: diet L1 category → cFMD category ────────────────────────────
DIET_L1_TO_CFMD_CATEGORY = {
    # Dairy
    "milk and milk products"            : "dairy",
    "dairy"                             : "dairy",
    "cheese"                            : "dairy",
    "yogurt"                            : "dairy",
    # Meat
    "meat poultry and game"             : "meat",
    "meat"                              : "meat",
    "poultry"                           : "meat",
    # Fish
    "fish and shellfish"                : "fish",
    "fish"                              : "fish",
    "seafood"                           : "fish",
    # Fruits and vegetables
    "vegetables and vegetable products" : "fruits_and_vegetables",
    "fruit"                             : "fruits_and_vegetables",
    "fruits"                            : "fruits_and_vegetables",
    "vegetables"                        : "fruits_and_vegetables",
    # Fermented grains
    "grain products"                    : "fermented_grains",
    "grains"                            : "fermented_grains",
    "bread"                             : "fermented_grains",
    "cereals and cereal products"       : "fermented_grains",
    # Alcohol / fermented beverages
    "alcoholic beverages"               : "alcohol",
    "alcohol"                           : "alcohol",
    "beverages"                         : "fermented_beverages",
    "nonalcoholic beverages"            : "fermented_beverages",
    # Legumes
    "legumes and legume products"       : "fermented_legumes",
    "legumes"                           : "fermented_legumes",
    "beans"                             : "fermented_legumes",
    # Seeds / nuts
    "nuts and seeds"                    : "fermented_seeds",
    "seeds"                             : "fermented_seeds",
    # Other / unmappable
    "fats and oils"                     : "other",
    "sugars and sweets"                 : "other",
    "soups sauces and gravies"          : "other",
    "mixed dishes"                      : "other",
    "snacks"                            : "other",
    "eggs"                              : "other",
    "spices and herbs"                  : "other",
    "unknown"                           : "unmapped",
}

dietary_consumption_matrix["cfmd_category"] = (
    dietary_consumption_matrix["diet_l1_category"]
    .map(DIET_L1_TO_CFMD_CATEGORY)
    .fillna("unmapped")
)

n_mapped   = (dietary_consumption_matrix["cfmd_category"] != "unmapped").sum()
n_unmapped = (dietary_consumption_matrix["cfmd_category"] == "unmapped").sum()
print(f"Food items mapped to a cFMD category : {n_mapped} / {len(dietary_consumption_matrix)}")
print(f"Food items unmapped                  : {n_unmapped}")


In [ ]:
# ── Attach FME weight to each food item ─────────────────────────────────────────
dietary_consumption_matrix["fme_weight"] = (
    dietary_consumption_matrix["cfmd_category"]
    .map(cfmd_category_weight_normalized.to_dict())
    .fillna(0.0)   # unmapped items get weight = 0
)

mapping_summary = (
    dietary_consumption_matrix
    .groupby(["diet_l1_category", "cfmd_category", "fme_weight"])
    .size()
    .reset_index(name="n_food_items")
    .sort_values("n_food_items", ascending=False)
)
print("Mapping summary (diet L1 → cFMD category → weight):")
display(mapping_summary)


---
## Section 6 – FME Score Calculation

**Formula:**
$$\text{FME}_{sample} = \sum_{\text{food items}} \text{consumption}_{\text{food, sample}} \times w_{\text{cfmd\_category(food)}}$$

Where:
- $\text{consumption}_{\text{food, sample}}$ = amount consumed (from `diet_raw`, grams or servings)
- $w$ = normalized cFMD microbial richness weight for the food's category

**Important:** Each `sample` corresponds to **one dietary day for one participant**.
The result is a score per fecal sample — not per participant.
Participant-level scores are derived later by averaging across their samples.


In [ ]:
# ── Subset diet matrix to analytical samples only ───────────────────────────────
diet_abundance_matrix = (
    dietary_consumption_matrix[analytical_sample_ids]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0.0)
)

fme_weight_vector = dietary_consumption_matrix["fme_weight"].values  # shape: (n_food_items,)

# ── Compute FME score: weighted sum of food consumption per sample ───────────────
# Result: one FME score per fecal sample (= one dietary day per participant)
fme_score_per_sample = (
    diet_abundance_matrix
    .multiply(fme_weight_vector, axis=0)
    .sum(axis=0)
    .rename("fme_score_daily")
)
fme_score_per_sample.index.name = "fecal_sample_id"

print(f"FME scores computed for : {len(fme_score_per_sample)} fecal samples")
print(f"Score range             : {fme_score_per_sample.min():.4f} – {fme_score_per_sample.max():.4f}")
print(f"Score mean ± std        : {fme_score_per_sample.mean():.4f} ± {fme_score_per_sample.std():.4f}")
print(f"Non-zero scores         : {(fme_score_per_sample > 0).sum()}")
print()
display(fme_score_per_sample.reset_index().head(8))


In [ ]:
# ── Participant-level FME summary (aggregate across samples) ────────────────────
# Note: samples are NOT independent — they come from the same 34 participants.
# Participant-level values are used ONLY for descriptive summaries,
# NOT as independent data points in statistical models.

fme_with_participant = pd.DataFrame({
    "fecal_sample_id" : fme_score_per_sample.index,
    "fme_score_daily" : fme_score_per_sample.values,
})
fme_with_participant["participant_id"] = (
    fecal_sample_metadata
    .loc[fme_with_participant["fecal_sample_id"], "UserName"]
    .values
)
fme_with_participant["study_day"] = (
    fecal_sample_metadata
    .loc[fme_with_participant["fecal_sample_id"], "StudyDayNo"]
    .values
)

# Participant-level summary (descriptive only)
participant_fme_summary = (
    fme_with_participant
    .groupby("participant_id")["fme_score_daily"]
    .agg(
        fme_mean     = "mean",
        fme_std      = "std",
        fme_min      = "min",
        fme_max      = "max",
        n_diet_days  = "count"
    )
    .reset_index()
)

print("Participant-level FME summary (descriptive — 34 participants):")
display(participant_fme_summary.sort_values("fme_mean", ascending=False).head(10))


In [ ]:
# ── Visualize FME scores ───────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Distribution of daily FME scores across all fecal samples
axes[0].hist(fme_score_per_sample.values, bins=30,
             color="steelblue", edgecolor="white")
axes[0].axvline(fme_score_per_sample.mean(), color="red", linestyle="--",
                label=f"Mean = {fme_score_per_sample.mean():.2f}")
axes[0].set_title("Daily FME Score Distribution\n(475 fecal samples)")
axes[0].set_xlabel("FME Score")
axes[0].set_ylabel("Number of Samples")
axes[0].legend()

# Mean FME per participant (sorted)
sorted_participants = participant_fme_summary.sort_values("fme_mean", ascending=False)
axes[1].bar(range(len(sorted_participants)), sorted_participants["fme_mean"],
            color="darkorange", edgecolor="none", alpha=0.85)
axes[1].set_title(f"Mean FME Score per Participant\n({len(sorted_participants)} participants)")
axes[1].set_xlabel("Participant (sorted by mean FME)")
axes[1].set_ylabel("Mean Daily FME Score")
axes[1].set_xticks([])

# FME over study timeline (mean per study day)
fme_by_day = fme_with_participant.groupby("study_day")["fme_score_daily"].mean()
axes[2].plot(fme_by_day.index, fme_by_day.values,
             marker="o", color="teal", linewidth=1.5, markersize=4)
axes[2].set_title("Mean FME Score Over Study Timeline")
axes[2].set_xlabel("Study Day")
axes[2].set_ylabel("Mean FME Score")

plt.suptitle("Food Microbiome Exposure (FME) Score — Overview", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()


---
## Section 7 – Gut Microbiome Alpha Diversity per Fecal Sample

Compute **Shannon entropy** for each fecal sample as a measure of gut microbial richness and evenness.
This is the primary outcome variable (composition proxy) at the sample level.


In [ ]:
# ── Shannon entropy per fecal sample ────────────────────────────────────────────
def compute_shannon_entropy(abundance_column):
    """Shannon entropy H = -Σ p_i * log(p_i) for non-zero taxa."""
    positive_values = abundance_column[abundance_column > 0]
    if positive_values.sum() == 0:
        return np.nan
    proportions = positive_values / positive_values.sum()
    return float(-np.sum(proportions * np.log(proportions)))

gut_microbiome_analytical = gut_microbiome_abundance[analytical_sample_ids]
shannon_diversity_per_sample = (
    gut_microbiome_analytical
    .apply(compute_shannon_entropy, axis=0)
    .rename("shannon_diversity")
)
shannon_diversity_per_sample.index.name = "fecal_sample_id"

print(f"Shannon diversity computed for : {shannon_diversity_per_sample.notna().sum()} samples")
print(f"Range  : {shannon_diversity_per_sample.min():.3f} – {shannon_diversity_per_sample.max():.3f}")
print(f"Mean   : {shannon_diversity_per_sample.mean():.3f}")


---
## Section 8 – Gut Microbiome Stability per Participant

**Stability** = how consistent is a participant's gut microbiome across their sampling days?

We use the **coefficient of variation (CV)** of Shannon diversity across a participant's samples:
$$CV_{participant} = \frac{\sigma(\text{Shannon})}{\mu(\text{Shannon})}$$

Lower CV = more stable microbiome.

> This is computed at the **participant level** (34 values) and joined back to the sample-level table as a per-participant attribute.


In [ ]:
# ── Map each fecal sample to its participant ─────────────────────────────────────
sample_to_participant = (
    fecal_sample_metadata
    .loc[analytical_sample_ids, "UserName"]
    .rename("participant_id")
)

shannon_with_participant = pd.DataFrame({
    "fecal_sample_id" : shannon_diversity_per_sample.index,
    "shannon_diversity": shannon_diversity_per_sample.values,
}).set_index("fecal_sample_id")
shannon_with_participant["participant_id"] = sample_to_participant

# ── Participant-level stability (CV of Shannon across their samples) ───────────
participant_gut_stability = (
    shannon_with_participant
    .groupby("participant_id")["shannon_diversity"]
    .agg(
        participant_shannon_mean = "mean",
        participant_shannon_std  = "std",
        participant_n_samples    = "count"
    )
    .assign(
        participant_shannon_cv = lambda df:
            df["participant_shannon_std"] / df["participant_shannon_mean"]
    )
    .reset_index()
)

print(f"Gut stability computed for {len(participant_gut_stability)} participants")
print(f"Shannon CV range : {participant_gut_stability['participant_shannon_cv'].min():.4f} – "
      f"{participant_gut_stability['participant_shannon_cv'].max():.4f}")
display(participant_gut_stability.sort_values("participant_shannon_cv").head(8))


---
## Section 9 – Assemble the Statistical Analysis Dataset

Merge all variables into one clean table at the **fecal sample level**.
Each row = one fecal sample = one dietary day for one participant.

| Column group | Columns | Role in statistical model |
|---|---|---|
| Identity | `fecal_sample_id`, `participant_id`, `study_day` | Grouping / random effect |
| Demographics | `Age`, `BMI`, `Gender` | Fixed-effect covariates |
| **FME score** | `fme_score_daily` | **Main predictor** |
| **Gut diversity** | `shannon_diversity` | **Main outcome** |
| **Gut stability** | `participant_shannon_cv` | **Secondary outcome** |
| Nutrition | `KCAL`, `FIBE`, `PROT`, `TFAT`, `CARB` | Dietary confounders |


In [ ]:
# ── Step 1: Base table — fecal sample IDs ──────────────────────────────────────
statistical_analysis_dataset = pd.DataFrame(
    index=pd.Index(analytical_sample_ids, name="fecal_sample_id")
)

# ── Step 2: Participant identity and demographics ───────────────────────────────
participant_meta_cols = ["UserName", "StudyDayNo", "Gender", "Age", "BMI",
                         "Weight", "Supplement", "Medications"]
participant_meta_cols = [c for c in participant_meta_cols
                         if c in fecal_sample_metadata.columns]

statistical_analysis_dataset = statistical_analysis_dataset.join(
    fecal_sample_metadata[participant_meta_cols]
    .rename(columns={"UserName": "participant_id", "StudyDayNo": "study_day"}),
    how="left"
)

# ── Step 3: Daily FME score ─────────────────────────────────────────────────────
statistical_analysis_dataset = statistical_analysis_dataset.join(
    fme_score_per_sample.rename("fme_score_daily"),
    how="left"
)

# ── Step 4: Gut alpha diversity (Shannon) ──────────────────────────────────────
statistical_analysis_dataset = statistical_analysis_dataset.join(
    shannon_diversity_per_sample.rename("shannon_diversity"),
    how="left"
)

# ── Step 5: Participant-level gut stability (CV) ────────────────────────────────
statistical_analysis_dataset = statistical_analysis_dataset.join(
    participant_gut_stability.set_index("participant_id")[
        ["participant_shannon_cv", "participant_shannon_mean", "participant_n_samples"]
    ],
    on="participant_id",
    how="left"
)

# ── Step 6: Daily nutrition totals ─────────────────────────────────────────────
nutrition_cols_keep = ["KCAL", "PROT", "TFAT", "CARB", "FIBE",
                       "SUGR", "SODI", "D_TOTAL", "D_YOGURT", "D_CHEESE",
                       "PF_MEAT", "PF_SEAFD_HI", "G_WHOLE"]
nutrition_cols_keep = [c for c in nutrition_cols_keep
                       if c in daily_nutrition_totals.columns]

statistical_analysis_dataset = statistical_analysis_dataset.join(
    daily_nutrition_totals[nutrition_cols_keep],
    how="left"
)

print(f"Statistical dataset shape  : {statistical_analysis_dataset.shape}")
print(f"Fecal samples (rows)       : {statistical_analysis_dataset.shape[0]}")
print(f"Variables (columns)        : {statistical_analysis_dataset.shape[1]}")
print(f"Participants represented   : {statistical_analysis_dataset['participant_id'].nunique()}")
print(f"Complete rows (no NaN)     : {statistical_analysis_dataset.dropna().shape[0]}")


In [ ]:
# ── Missing value report ───────────────────────────────────────────────────────
missing_report = (
    statistical_analysis_dataset.isnull().sum()
    .reset_index()
    .rename(columns={"index": "variable", 0: "n_missing"})
)
missing_report["pct_missing"] = (
    missing_report["n_missing"] / len(statistical_analysis_dataset) * 100
).round(1)
missing_report = missing_report[missing_report["n_missing"] > 0].sort_values("n_missing", ascending=False)

if missing_report.empty:
    print("No missing values in the statistical dataset.")
else:
    print("Missing values:")
    display(missing_report)


In [ ]:
# ── Descriptive statistics ─────────────────────────────────────────────────────
key_vars = ["fme_score_daily", "shannon_diversity",
            "participant_shannon_cv", "Age", "BMI", "KCAL", "FIBE"]
key_vars = [v for v in key_vars if v in statistical_analysis_dataset.columns]

print("Descriptive statistics — key variables:")
display(statistical_analysis_dataset[key_vars].describe().T.round(3))


In [ ]:
# ── Quick scatter: FME score vs. gut Shannon diversity ─────────────────────────
plot_df = statistical_analysis_dataset.dropna(
    subset=["fme_score_daily", "shannon_diversity"]
)

plt.figure(figsize=(7, 5))
plt.scatter(plot_df["fme_score_daily"], plot_df["shannon_diversity"],
            alpha=0.5, s=25, c="teal", edgecolors="none")
if len(plot_df) > 2:
    z = np.polyfit(plot_df["fme_score_daily"], plot_df["shannon_diversity"], 1)
    x_line = np.linspace(plot_df["fme_score_daily"].min(),
                         plot_df["fme_score_daily"].max(), 100)
    plt.plot(x_line, np.poly1d(z)(x_line), "r--", lw=1.5, label="Trend line")
plt.xlabel("Daily FME Score (food microbiome exposure)")
plt.ylabel("Shannon Diversity (gut microbiome)")
plt.title("FME Score vs. Gut Alpha Diversity\n(475 fecal samples from 34 participants)")
plt.legend()
plt.tight_layout()
plt.show()


---
## Section 10 – Save Outputs

In [ ]:
# ── Save statistical analysis dataset ─────────────────────────────────────────
statistical_analysis_dataset.to_csv(DATA_DIR / "fme_statistical_dataset.csv")
participant_gut_stability.to_csv(DATA_DIR / "participant_gut_stability.csv", index=False)
fme_with_participant.to_csv(DATA_DIR / "fme_sample_level_scores.csv", index=False)

print("Saved:")
print(f"  fme_statistical_dataset.csv    {statistical_analysis_dataset.shape}  — main analytical table")
print(f"  participant_gut_stability.csv  {participant_gut_stability.shape}  — stability per participant")
print(f"  fme_sample_level_scores.csv    {fme_with_participant.shape}  — FME scores with participant labels")


---
## Summary

### What was built
The `fme_statistical_dataset.csv` table is the **primary input for all statistical analyses**.

### Structural note (important for modelling)
- **Unit of observation**: fecal sample (475 rows)
- **Grouping variable**: `participant_id` (34 participants) → must be included as a **random effect**
- **Main predictor**: `fme_score_daily`
- **Main outcomes**: `shannon_diversity` (composition), `participant_shannon_cv` (stability)

### Recommended next steps
```
Mixed-effects model (composition):
  shannon_diversity ~ fme_score_daily + Age + BMI + KCAL + (1 | participant_id)

Mixed-effects model (stability — participant level, n=34):
  participant_shannon_cv ~ mean(fme_score_daily) + Age + BMI
```
